# Libraries and global variables

In [ ]:
from pathlib import Path
import json
import sys
from typing import Optional
#import config as cfg

import numpy as np
import pandas as pd
from PIL import Image

from fastai.vision.all import(
    DataBlock, ImageBlock, BBoxBlock, BBoxLblBlock,
    ColReader, ColSplitter,
    RandomSplitter, get_image_files,
    Resize, DataLoaders
)
# from fastai.data.transforms import parent_label

from fastai.vision.augment import (
    Brightness, Contrast, aug_transforms
)
from fastai.vision.core import imagenet_stats

## Directory layout

In [ ]:
PLATFORM = "colab " # "colab", "kaggle", "local"

if PLATFORM == "colab":
  ROOT = Path("/content/project_gn_food_estimator")
elif PLATFORM == "kaggle":
  ROOT = Path("/kaggle/working/project_gn_food_estimator")
else:
  ROOT = Path(__file__).resolve().parent

In [ ]:
DATA_DIR      = ROOT / "data"
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXPORTS_DIR   = DATA_DIR / "exports"
MODELS_DIR    = ROOT / "models"
TRAINING_DIR  = ROOT / "training"

In [ ]:
# Labels file
COCO_MODEL1_JSON = EXPORTS_DIR / "model1_detection.json"

In [ ]:
# Model checkpoint path
MODEL1_PATH = MODELS_DIR / "model1_detector.pth"

## Parameters

In [ ]:
# Dataset split ratios
SPLIT_TRAIN = 0.80
SPLIT_VAL   = 0.10
SPLIT_TEST  = 0.10
RANDOM_SEED = 42

In [ ]:
# -----------------------------------------------------------------------------
# GN container standard specifications
# Two containers currently in scope: GN 1/1 (large) and GN 1/2 (small).
#
# Dimensions source: EN 631-1 standard
# Volume = inner_length × inner_width × depth (all mm → litres)
#
# Key:
#   gn_id       : string label used throughout the pipeline
#   label       : human-readable name (matches Model #1 class labels)
#   outer_l_mm  : outer length (mm)
#   outer_w_mm  : outer width (mm)
#   inner_l_mm  : inner length (mm)
#   inner_w_mm  : inner width (mm)
#   depth_mm    : internal depth (mm) — 65 mm default depth
#   volume_l    : nominal volume in litres
# -----------------------------------------------------------------------------
GN_CONTAINERS = {
    "GN_1_1": {
        "gn_id":      "GN_1_1",
        "label":      "large container",
        "outer_l_mm": 530,
        "outer_w_mm": 325,
        "inner_l_mm": 520,
        "inner_w_mm": 312,
        "depth_mm":   65,
        "volume_l":   round(520 * 312 * 65 / 1e6, 2),   # ≈ 10.55 L
    },
    "GN_1_2": {
        "gn_id":      "GN_1_2",
        "label":      "small container",
        "outer_l_mm": 325,
        "outer_w_mm": 265,
        "inner_l_mm": 312,
        "inner_w_mm": 254,
        "depth_mm":   65,
        "volume_l":   round(312 * 254 * 65 / 1e6, 2),   # ≈ 5.15 L
    },
}

In [ ]:
# Snap-to-standard tolerance: accept pixel-derived volume within this margin
GN_SNAP_TOLERANCE = 0.10   # 10% — snap to nearest GN if within this fraction

In [ ]:
# Model #1 class labels (must match Label Studio annotation labels exactly)
MODEL1_CLASSES = ["small container", "large container", "ArUco marker"]

## Training hyperparameters

In [ ]:
# Model #1 — ResNet-50, object detection head
MODEL1_BACKBONE        = "resnet50"
MODEL1_BATCH_SIZE      = 8
MODEL1_EPOCHS_FROZEN   = 5       # Phase 1: head only
MODEL1_EPOCHS_UNFROZEN = 10      # Phase 2: full fine-tune
MODEL1_LR_FROZEN       = 1e-3
MODEL1_LR_UNFROZEN     = 1e-4
MODEL1_MIXED_PREC      = True    # fp16

# Loading data

Dataset loader for:
- Model #1 — object detection (COCO JSON bounding-box annotations)

The function return a fastai DataLoaders object ready for training.

Label Studio COCO export format assumptions:
- Detection export  : standard COCO format with "images", "annotations", "categories" keys. Each annotation has bbox in [x_min, y_min, width, height] format.

Split strategy:
- Splits are performed at the image level (not annotation level) to avoid data leakage between train / val / test.
- Ratios: 80 / 10 / 10
- Random seed is fixed for reproducibility.

In [ ]:
# Shared helpers

def _load_coco_json(json_path: Path) -> dict:
    """
    Load and validate a COCO JSON file exported from Label Studio.
    """
    if not json_path.exists():
        raise FileNotFoundError(
            f"COCO annotation file not found: {json_path}\n"
            f"Export your Label Studio project as COCO JSON and place it at "
            f"the path defined in {COCO_MODEL1_JSON}."
        )
    with open(json_path, "r") as f:
        data = json.load(f)
    required_keys = {"images", "annotations", "categories"}
    if not required_keys.issubset(data.keys()):
        raise ValueError(
            f"COCO JSON at {json_path} is missing keys: "
            f"{required_keys - data.keys()}"
        )
    return data


def _split_image_ids(
        image_ids: list[int],
        train_ratio: float = SPLIT_TRAIN,
        val_ratio: float = SPLIT_VAL,
        seed: int = RANDOM_SEED,
    ) -> tuple[set[int], set[int], set[int]]:
    """
    Randomly split a list of image IDs into train / val / test sets.

    Returns:
        tuple[set, set, set]
            (train_ids, val_ids, test_ids)
    """
    rng = np.random.default_rng(seed)
    ids = np.array(image_ids)
    rng.shuffle(ids)

    n = len(ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_ids = set(ids[:n_train].tolist())
    val_ids = set(ids[n_train:n_train + n_val].tolist())
    test_ids = set(ids[n_train + n_val:].tolist())

    print(
        f"Split -> train: {len(train_ids)} | val: {len(val_ids)} | test: {len(test_ids)} images"
    )
    return train_ids, val_ids, test_ids

In [ ]:
# DataLoaders

def get_detection_dataloaders(
        coco_json_path: Path = COCO_MODEL1_JSON,
        image_dir: Path = RAW_DIR,
        image_size: int = IMAGE_SIZE,
        batch_size: int = MODEL1_BATCH_SIZE,
        train_tfms: list = None,
        device: str = "cuda",
        verbose: bool = True
    ) -> tuple[DataLoaders, pd.DataFrame, pd.DataFrame]:
    """
    Build fastai DataLoaders for bounding-box object detection (model #1).

    Parses a Label Studio COCO JSON export and constructs a DataFrame with one
    row per annotation, then hands it to a fastai DataBlock.

    Parameters:
        coco_json_path : Path
            Path to the COCO JSON annotation file.
        image_dir : Path
            Directory containing the raw images.
        image_size : int
            Target image size (longer edge).
        batch_size : int
            Training batch size.
        train_tfms : list, optional
            Fastai transforms for training set. If None, uses aug_transforms defaults.
        device : str
            "cuda" or "cpu"
        verbose : bool
            Print split and class information.

    Returns:
        tuple[DataLoaders, pd.DataFrame, pd.DataFrame]
            (dls, train_df, test_df) - test_df is returned separately for
            post-training evaluation; it is NOT included in the DataLoaders.
    """
    if verbose:
        print(f"Loading COCO annonations from: {coco_json_path}")

    coco = _load_coco_json(coco_json_path)

    # Build category ID -> label name mapping
    id_to_label = {cat["id"]: cat["name"] for cat in coco["categories"]}
    if verbose:
        print(f"Categories: {id_to_label}")

        # Warn if labels do not match config
        coco_labels = set(id_to_label.values())
        cfg_labels = set(MODEL1_CLASSES)
        if coco_labels != cfg_labels:
            print(
                f"WARNING: COCO labels {coco_labels} differ from "
                f"configuration {cfg_labels}."
                f"Update MODEL1_CLASSES if needed."
            )

    # Build image ID -> filename mapping
    id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
    all_images_ids = list(id_to_file.keys())

    # Split image IDs
    if verbose:
        print(f"Total images: {len(all_image_ids)}")
    train_ids, val_ids, test_ids = _split_image_ids(all_image_ids)

    # Build annotation DataFrame
    # COCO bbox: [x_min, y_min, width, height] -> fastai want [x_min, y_min, x_max, y_max]
    rows = []
    for ann in coco["annotations"]:
        img_id = ann["image_id"]
        filename = id_to_file[img_id]
        x, y, w, h = ann["bbox"]
        label = id_to_label[ann["category_id"]]
        split = (
            "train" if img_id in train_ids else
            "valid" if img_id in val_ids else
            "test"
        )
        rows.append({
            "image_path": str(image_dir / filename),
            "x_min": x,
            "y_min": y,
            "x_max": x + w,
            "y_max": y + h,
            "label": label,
            "split": split
        })

    full_df = pd.DataFrame(rows)

    # Separate test set - not fed into DataLoaders
    test_df = full_df[full_df["split"] == "test"].copy()
    train_df = full_df[full_df["split"] != "test"].copy()
    train_df["is_valid"] = train_df["split"] == "valid"

    if verbose:
        print(f"Annotation rows -> train + val: {len(train_df)} | test: {len(test_df)}")

    # Aggregate annotations per image into lists (fastai bbox format)
    def _agg_bboxes(grp):
        bboxes = grp[["x_min", "y_min", "x_max", "y_max"]].values.tolist()
        labels = grp["label"].tolist()
        return pd.Series({
            "image_path": grp["image_path"].iloc[0],
            "bboxes": bbboxes,
            "labels": labels,
            "is_valid": grp["is_valid"].iloc[0]
        })


    agg_df = (
        train_df
        .groupby("image_path", group_keys = False)
        .apply(_agg_bboxes)
        .reset_index(drop=True)
    )

    # Build fastai DataBlock
    # Note: BBoxBlock + BBoxLblBlock require paired getters
    dblock = DataBlock(
        blocks = (ImageBlock, BBoxBlock, BBoxLblBlock(add_na = True)),
        n_inp = 1,
        get_x = ColReader("image_path")
        get_y = [ColReader("bboxes"), ColReader("labels")],
        splitter = ColSplitter("is_valid"),
        item_tfms = Resize(image_size),
        batch_tfms = train_tfms
    )

    dls = dblock.dataloaders(
        agg_df,
        bs = batch_size,
        devive = device
    )

    if verbose:
        print(f"DataLoaders ready - vocab: {dls.vocab}")

    return dls, train_df, test_df

# Data Preprocessing

In [ ]:
# Image pre-processing
IMAGE_SIZE = 512
NORM_MEAN = [0.485, 0.456, 0.406] # ImageNet statistics
NORM_STD = [0.229, 0.224, 0.225]
HIST_EQ_ENABLED = True

## Data augmentation functions

Augmentationt pipelines for Model #1. Each function returns a fastai Transform list that can be passed directly to a DataBlock or DataLoaders.

Augmentations applied (all toggleable):
- brightness
- rotation
- blur
- scaling
- flipping
- contrast

Design note: top-down food photography has specific characteristics:
- No meaningful vertical flip (gravity / food surface matters)
- Rotation up to ±30° is realistic (user hand angle variation)
- Moderate blur simulates slight camera shake at 25-28 cm distance
- Brightness / contrast variation simulates kitchen lighting changes

In [ ]:
# Data augmentation toggles / hyperparameters
AUG_BRIGHTNESS = True
AUG_ROTATION   = True
AUG_BLUR       = True
AUG_SCALING    = True
AUG_FLIPPING   = True
AUG_CONTRAST   = True

In [ ]:
def get_detection_transforms(size: int = IMAGE_SIZE):
    """
    Returns (train_tfms, valid_tfms) for Model #1 (object detection).
    """

# Dataset or batch

# Training

## Find learning rate

## Fine tune and actual training

# Results

# Save trained model

# Load trained model

## Export model for production

# Predictions / Inference

## Save predictions